# Silver Enriched: Product Dimension (SCD Type 2)

**Job Name:** `product_dim_scd2`  
**Source:** `bronze.orders_raw` (via CDF)  
**Target:** `silver_enriched.product_dim`  
**SCD2 Trigger:** Change in `product_name`, `brand`, or `category`

## Import Utils

In [ ]:
import sys
sys.path.append("/opt/spark-notebooks/silver_enriched")

from utils import (
    get_last_processed_version,
    save_last_processed_version,
    read_bronze_cdf,
    get_current_bronze_version
)

print("Utils imported ✅")

## Create Table

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver_enriched")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_enriched.product_dim (
        product_sk BIGINT,
        product_id STRING,
        product_name STRING,
        brand STRING,
        category STRING,
        effective_from TIMESTAMP,
        effective_to TIMESTAMP,
        is_current BOOLEAN
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/enriched/product_dim'
""")

print("Table ready ✅")

## Check Last Processed Version

In [ ]:
JOB_NAME = "product_dim_scd2"

last_version = get_last_processed_version(spark, JOB_NAME)
print(f"Last processed version: {last_version}")

## Read Only New Data from Bronze (CDF)

In [ ]:
bronze_df = read_bronze_cdf(spark, last_version)
new_row_count = bronze_df.count()
print(f"New rows to process: {new_row_count}")

## Parse & Deduplicate Product Attributes

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

if new_row_count > 0:
    product_schema = StructType([
        StructField("order", StructType([
            StructField("order_time", StringType()),
            StructField("items", ArrayType(StructType([
                StructField("product_details", StructType([
                    StructField("product_id", StringType()),
                    StructField("product_name", StringType()),
                    StructField("brand", StringType()),
                    StructField("category", StringType())
                ]))
            ])))
        ]))
    ])

    parsed = bronze_df.withColumn("data", from_json(col("raw_payload"), product_schema))

    # Explode items array to get individual products
    incoming_products = parsed.select(
        explode(col("data.order.items")).alias("item"),
        col("data.order.order_time").cast("timestamp").alias("order_time")
    ).select(
        col("item.product_details.product_id").alias("product_id"),
        col("item.product_details.product_name").alias("product_name"),
        col("item.product_details.brand").alias("brand"),
        col("item.product_details.category").alias("category"),
        col("order_time")
    )

    # Deduplicate: Keep the LATEST version per product_id
    w = Window.partitionBy("product_id").orderBy(col("order_time").desc())

    latest_products = (
        incoming_products
        .withColumn("rn", row_number().over(w))
        .filter("rn = 1")
        .drop("rn", "order_time")
    )

    latest_products.createOrReplaceTempView("incoming_products")
    print(f"Unique products to process: {latest_products.count()}")
    latest_products.show(truncate=False)
else:
    print("No new data. Skipping. ✅")

## SCD2 MERGE

In [ ]:
if new_row_count > 0:
    existing_count = spark.sql("SELECT COUNT(*) FROM silver_enriched.product_dim").collect()[0][0]

    if existing_count == 0:
        print("Seeding dimension...")
        spark.sql("""
            INSERT INTO silver_enriched.product_dim
            SELECT
                monotonically_increasing_id() AS product_sk,
                product_id, product_name, brand, category,
                current_timestamp() AS effective_from,
                CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to,
                true AS is_current
            FROM incoming_products
        """)
        print("Dimension seeded ✅")

    else:
        print("Running SCD2 merge...")
        spark.sql("""
            MERGE INTO silver_enriched.product_dim AS target
            USING incoming_products AS source
            ON target.product_id = source.product_id AND target.is_current = true
            WHEN MATCHED AND (
                target.product_name != source.product_name OR
                target.brand        != source.brand        OR
                target.category     != source.category
            )
            THEN UPDATE SET
                target.is_current   = false,
                target.effective_to = current_timestamp()
        """)
        print("Step 1 done: Closed old versions.")

        spark.sql("""
            INSERT INTO silver_enriched.product_dim
            SELECT
                (SELECT COALESCE(MAX(product_sk), 0) FROM silver_enriched.product_dim)
                    + monotonically_increasing_id() + 1 AS product_sk,
                src.product_id, src.product_name, src.brand, src.category,
                current_timestamp() AS effective_from,
                CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to,
                true AS is_current
            FROM incoming_products src
            LEFT ANTI JOIN silver_enriched.product_dim dim
                ON src.product_id = dim.product_id AND dim.is_current = true
        """)
        print("Step 2 done: Inserted new versions ✅")

    current_version = get_current_bronze_version(spark)
    save_last_processed_version(spark, JOB_NAME, current_version, new_row_count)

## Verify

In [ ]:
spark.sql("""
    SELECT product_sk, product_id, product_name, brand, category,
           is_current, effective_from, effective_to
    FROM silver_enriched.product_dim
    ORDER BY product_id, effective_from
""").toPandas()